In [32]:
#URL : https://www.kaggle.com/competitions/nlp-getting-started/overview

In [49]:
import os
import numpy as np
import pandas as pd
import re
import unicodedata
import warnings

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split,cross_validate,StratifiedKFold
from sklearn.metrics import(
    classification_report,
    f1_score,precision_score,recall_score,accuracy_score
)
from sklearn.feature_extraction.text import TfidfVectorizer

#model
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

#virtualize
import seaborn as sns

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

warnings.simplefilter(action = 'ignore')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
VOCAB_SIZE = 5000
sample_input = {
    'keyword' : ['ablaze','accident'],
    'location' : ['Birmingham','Arlington, TX'],
    'text' : ['@bbcmtd Wholesale Markets ablaze http://t.co/lHYXEOHY6C',
              "#TruckCrash Overturns On #FortWorth Interstate http://t.co/Rs22LJ4qFp Click here if you've been in a crash&gt;http://t.co/Ld0unIYw4k"],
    'target' : [1,1]
}

In [54]:
X_train = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/nlp-getting-started/train.csv',index_col = 'id')
X_train.columns

Index(['keyword', 'location', 'text', 'target'], dtype='object')

In [55]:
y_train = X_train['target']
X_train.drop(columns = 'target',inplace = True)

X_train,X_valid,y_train,y_valid = train_test_split(X_train,y_train,test_size = 0.2,stratify = y_train,random_state = 42)

In [56]:
for col in X_train.columns:
    print(f"{col} : {pd.unique(X_train[col]).shape[0]}")

keyword : 222
location : 2809
text : 6018


In [57]:
def normalize(text,lower_case = True,remove_punctuation = True,replace_num = True):
    text = unicodedata.normalize('NFKC',text)
    text = ' '.join(text.split())
    
    if lower_case:
        text = text.lower()
    if replace_num:
        text = re.sub(r'\d+','<NUM>',text)
    if remove_punctuation:
        text = re.sub(r"[^\w\s!?.,#()@<>'\"]",'',text)
    return text

def preprocess(text,windows_size = 3):
    text = normalize(text)
    tokens = word_tokenize(text)
    
    negative_words = {'not',"n't",'no','neither','never','nobody','nothing',
                      "isn't","don't","doesn't"}
    punctuation_words = {',','.',';'}
    contrast_words = {'and','so','because','since','but','although','despite','moreover','due','though'}
    
    res = []
    counts = 0
    for token in tokens:
        if token in negative_words:
            counts = windows_size
            res.append(token)
        elif token in punctuation_words or token in contrast_words:
            res.append(token)
            counts = 0
        elif counts > 0:
            res.append(f"NOT_{token}")
            counts -= 1
        else:
            res.append(token)
    return ' '.join(res)

In [77]:
def preprocess_text_column(X):
    if isinstance(X, pd.DataFrame):
        X = X.iloc[:, 0]
    elif not isinstance(X, pd.Series):
        X = pd.Series(X)
    return X.astype(str).apply(preprocess)
feature_cols = ['keyword','location']
text_col = ['text']
full_preprocess = ColumnTransformer(transformers = [
    ('features',Pipeline(steps = [
        ('impute',SimpleImputer(strategy = 'constant',fill_value = 'NOT')),
        ('encode',OrdinalEncoder(handle_unknown = 'use_encoded_value',unknown_value = -1))
    ]),feature_cols),
    ('text',Pipeline(steps = [
        ('preprocess_text',FunctionTransformer(preprocess_text_column,validate = False)),
        ('tfidf',TfidfVectorizer(
        ngram_range = (1,2),
        max_features = VOCAB_SIZE,
        min_df = 2,
        max_df =0.85,
        sublinear_tf = True))
    ]),text_col),
])
semi_preprocess = ColumnTransformer(transformers = [
    ('text',"passthrough",text_col),
    ('feature',SimpleImputer(strategy = 'constant',fill_value = 'NOT'),feature_cols)
])
model_list = {
    'catboost' : CatBoostClassifier(
            iterations = 500,
            learning_rate = 0.015,
            depth = 6,
            loss_function = "Logloss",
            thread_count = -1
        ),
    'XGB' : Pipeline(steps = [
        ('preprocess',full_preprocess),
        ('model',XGBClassifier(
            n_estimators = 500,
            max_depth = 6,
            learning_rate = 0.015,
            n_jobs = -1,
            random_state = 42
        ))
    ]),
    'Logistic' : Pipeline(steps = [
        ('preprocess',full_preprocess),
        ('model',LogisticRegression(
            max_iter = 1000,
            class_weight = 'balanced',
            random_state = 42
        ))
    ])
}

In [83]:
for name,model in model_list.items():

    y_preds = 0
    if name == 'catboost':
        X_train_score = semi_preprocess.fit_transform(X_train)
        model.fit(X_train_score,y_train,cat_features = [0,1],text_features = [2])
        X_valid_score = semi_preprocess.transform(X_valid)
        y_preds = model.predict(X_valid_score)
    else:
        model.fit(X_train,y_train)
        y_preds = model.predict(X_valid)
    
    acc = accuracy_score(y_valid,y_preds)
    f1 = f1_score(y_valid,y_preds)
    precision = precision_score(y_valid,y_preds)
    recall = recall_score(y_valid,y_preds)
    
    print(f"{'=' * 50}")
    print(f"TRANGING {name}")
    
    print(f"Accuracy: {acc:.4f}")
    print(f"F1: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    #print(classification_report(y_valid,y_preds,target_names = ['Negative', 'Positive']))

0:	learn: 0.6897252	total: 47.4ms	remaining: 23.6s
1:	learn: 0.6867714	total: 91.5ms	remaining: 22.8s
2:	learn: 0.6838569	total: 131ms	remaining: 21.8s
3:	learn: 0.6809689	total: 170ms	remaining: 21.1s
4:	learn: 0.6780181	total: 209ms	remaining: 20.7s
5:	learn: 0.6753212	total: 249ms	remaining: 20.5s
6:	learn: 0.6724805	total: 292ms	remaining: 20.6s
7:	learn: 0.6700482	total: 331ms	remaining: 20.4s
8:	learn: 0.6676068	total: 370ms	remaining: 20.2s
9:	learn: 0.6655193	total: 410ms	remaining: 20.1s
10:	learn: 0.6630891	total: 451ms	remaining: 20.1s
11:	learn: 0.6607517	total: 492ms	remaining: 20s
12:	learn: 0.6584514	total: 532ms	remaining: 19.9s
13:	learn: 0.6562559	total: 573ms	remaining: 19.9s
14:	learn: 0.6541229	total: 613ms	remaining: 19.8s
15:	learn: 0.6518759	total: 655ms	remaining: 19.8s
16:	learn: 0.6499383	total: 695ms	remaining: 19.7s
17:	learn: 0.6482716	total: 706ms	remaining: 18.9s
18:	learn: 0.6461587	total: 746ms	remaining: 18.9s
19:	learn: 0.6443794	total: 787ms	remaini

In [85]:
X_test = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/nlp-getting-started/test.csv',index_col = 'id')
preds = model_list['Logistic'].predict(X_test)
submission = pd.DataFrame({
    'id' : X_test.index,
    'target' : preds
})
submission.to_csv('submission.csv',index = False)